# Tennis Court Keypoints Model: Train & Export to TFJS
Run this in Google Colab or your local Jupyter environment. It demonstrates how to export a `.pth` model to a format your web app can run (`.json` + `.bin`).

In [ ]:
!pip install tensorflowjs onnx onnxruntime

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.onnx
import onnx

# 1. Define your model architecture
# Assuming your PyTorch model outputs 4 corners -> 8 values (x,y per corner).
# If your dataset uses 14 points, change num_keypoints to 14.
class CourtKeypointModel(nn.Module):
    def __init__(self, num_keypoints=4):
        super(CourtKeypointModel, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        # Replace the last fully connected layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_keypoints * 2)
        
    def forward(self, x):
        return self.backbone(x)

# Load your trained .pth weights into the model
model = CourtKeypointModel(num_keypoints=4)
# model.load_state_dict(torch.load('my_court_keypoints.pth', map_location='cpu'))
model.eval() # Set to evaluation mode for export

## Training Loop (Optional)
If you still need to train your model, load your data here. Your targets should be normalized between `0.0` and `1.0` if possible (or pixel values divided by 640).

In [ ]:
# ... (Your training loop over Dataset/DataLoader) ...
# Example:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# criterion = nn.MSELoss()
# for epoch in range(epochs):
#     ...
print("Model trained.")

## Export to ONNX
TensorFlow.js cannot load `.pth` files directly. We must first export the model to ONNX, then use `tensorflowjs_converter` to change the ONNX model to the TFJS GraphModel format.

In [ ]:
# 2. Export to ONNX
dummy_input = torch.randn(1, 3, 640, 640) # Match the 640x640 input from your React app
onnx_path = "court_keypoints.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=12,          # Opset 12 is well supported by converters
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f"Exported to {onnx_path}")

## Convert ONNX to TensorFlow.js (Graph Model)
This gives us the frontend `.json` and `.bin`.

In [ ]:
# 3. Run the TFJS converter
# This will create a folder called 'tfjs_court_model' containing model.json and .bin files
!tensorflowjs_converter \
    --input_format=onnx \
    --output_format=tfjs_graph_model \
    --signature_name=serving_default \
    court_keypoints.onnx \
    tfjs_court_model

In [ ]:
import os
print("TFJS Model Files exported:")
print(os.listdir("tfjs_court_model"))
# You can now download tfjs_court_model/model.json and tfjs_court_model/group1-shard1of1.bin
# and upload them directly into the Web App using the Court Calibrator upload button!

# Note: Verify the index mapping. If your keypoints correspond to different areas
# (e.g. keypoint 0 is TL instead of BL), simply update the indices 
# in the Custom CNN block within `src/App.tsx`.